In [ ]:
import os
import sys
import re
import string
import random
import warnings
import argparse
import numpy as np
import pandas as pd
import torch
import time
import seaborn as sns
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from io import StringIO
from unicodedata import category
from bs4 import BeautifulSoup
from markdown import markdown
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score,classification_report
from torch.utils.data import DataLoader, RandomSampler, Dataset
from transformers import (
    BertTokenizer, BertForSequenceClassification, BertForMaskedLM,
    XLNetTokenizer, XLNetForSequenceClassification,
    RobertaTokenizer, RobertaForSequenceClassification, RobertaForMaskedLM,
    AlbertTokenizer, AlbertForSequenceClassification, AlbertForMaskedLM,
    get_scheduler, AdamW
)


drive.mount('/content/drive')


# Tokenlized

In [ ]:
# Regular expression for GitHub username mentions
USERNAME_REGEX = r"(\s|^)@(\S*\s?)"

# Generate Unicode punctuation set
punctuation = {chr(i) for i in range(sys.maxunicode + 1) if category(chr(i)).startswith(("P", "S"))}

# Dictionary to count token replacements
counters = {}

def remove_punctuation(text):
    """Remove all punctuation characters from the given text."""
    return "".join(char for char in text if char not in punctuation)

def clean_text(text):
    """Remove quoted text and large code blocks from GitHub issues or comments."""
    # Remove quoted text from emails/notifications
    text = re.sub(r"^(On[\s\S]*?notifications@github\.com\s*?wrote:\s*?)?(^(\>).*\s)*", '', text, flags=re.MULTILINE)

    # Remove code blocks enclosed in triple backticks
    text = re.sub(r"```[a-z]*\n[\s\S]*?\n```", "", text)

    return text

def replace_token(regex, token_name, text):
    """
    Replace matched patterns in the text with the specified token.

    Args:
        regex (str): The regular expression pattern to match.
        token_name (str): The replacement token name.
        text (str): The input text.

    Returns:
        tuple: (processed_text, number_of_replacements)
    """
    replaced_text, replacements = re.subn(regex, f" {token_name} ", text, flags=re.MULTILINE)
    counters[token_name] = counters.get(token_name, 0) + replacements
    return replaced_text, replacements

def tokenize_text(text):
    """
    Tokenizes a given text by replacing specific elements such as emails, mentions, URLs, etc.

    Args:
        text (str): The input text.

    Returns:
        tuple: (processed_text, total_replacements)
    """
    total_replacements = 0

    text, replacements = replace_token(r"\S+@\S*\s?", "MEMAIL", text)
    total_replacements += replacements

    text, replacements = replace_token(USERNAME_REGEX, "MMENTION", text)
    total_replacements += replacements

    text, replacements = replace_token(r"`([^`]*)`", "MICODE", text)
    total_replacements += replacements

    text, replacements = replace_token(r"\b\d+\.\d+(\.\d+)*\b", "MVERSIONNUMBER", text)
    total_replacements += replacements

    text, replacements = replace_token(r"(\s|^)#\d+", "MISSUEMENTION", text)
    total_replacements += replacements

    text, replacements = replace_token(
        r"([a-zA-Z0-9]+):\/\/([\w_-]+(?:\.[\w_-]+)*)[\w.,@?^=%&:\/~+#-]*[\w@?^=%&\/~+#-]",
        "MURL",
        text,
    )
    total_replacements += replacements

    return text, total_replacements

def remove_markdown_content(text):
    """
    Converts Markdown content to plain text by removing all Markdown formatting.

    Args:
        text (str): The input Markdown text.

    Returns:
        str: Cleaned text without Markdown formatting.
    """
    html = markdown(text)
    return "".join(BeautifulSoup(html, "lxml").findAll(text=True))

def transform_text(row):
    """
    Transforms a row by cleaning and tokenizing its text content.

    Args:
        row (dict): A dictionary containing a 'Text' key.

    Returns:
        tuple: (processed text, number of replacements)
    """
    text = row.get("Text", "")

    if not isinstance(text, str):
        warnings.warn(f"Converting non-string type to string: {type(text)}")
        text = str(text)

    text, replaced_count = tokenize_text(text)
    text = text.replace("\n", "")
    return text, replaced_count

##Usage

In [ ]:


# Define input dataset paths
input_paths = [
    "/content/drive/MyDrive/Software_Development_Sentiment_Classification/so-dataset.csv",
    "/content/drive/MyDrive/Software_Development_Sentiment_Classification/gh-dataset.csv",
    "/content/drive/MyDrive/Software_Development_Sentiment_Classification/crossplatform_sf_dataset.csv"
]

# Define the text transformation function (ensure transform_text is correctly implemented)
def transform_text(row):
    # Modify this function according to your needs
    # Example: return the original text and a dummy replacement count
    return row["Text"], 1

# Loop through each dataset and process it
for input_path in input_paths:
    # Generate output file name
    output_filename = os.path.splitext(os.path.basename(input_path))[0] + "_tokenized.csv"
    output_path = os.path.join(os.path.dirname(input_path), output_filename)

    # Load dataset
    df = pd.read_csv(input_path)
    print(f"Processing dataset: {input_path}")
    print(df.head())  # Print first few rows for verification

    # Apply text transformation
    df[["Text", "replaced_token"]] = df.apply(transform_text, axis=1, result_type="expand")

    # Calculate total replacements from `replaced_token` column
    total_replacements = df["replaced_token"].sum()

    # Save processed dataset
    df.to_csv(output_path, header=True, index=False)

    print(f"Tokenized dataset saved to: {output_path}\n")


Processing dataset: /content/drive/MyDrive/Software_Development_Sentiment_Classification/so-dataset.csv
                                                Text  Polarity
0  In this situation, when I click on the greyed ...         0
1  After that a download progress status with pro...         0
2  Change the double quotationCODE_FRAGMENT to to...         0
3      E.g. I get an array of CODE_FRAGMENT objects.         0
4  Then I tried my own implementation with CODE_F...         0
Total replacements for /content/drive/MyDrive/Software_Development_Sentiment_Classification/so-dataset.csv: 450
Tokenized dataset saved to: /content/drive/MyDrive/Software_Development_Sentiment_Classification/so-dataset_tokenized.csv

Processing dataset: /content/drive/MyDrive/Software_Development_Sentiment_Classification/gh-dataset.csv
                                                Text  Polarity
0          Guess there is a typo here: `translate`."         0
1  @arturoc: If you multiply-include `gst.h`, wou... 

# Dataset Overview: `crossplatform_sf_dataset.csv`

This dataset is designed for **Software Development Sentiment Classification**, containing user comments or discussions from different platforms with sentiment labels.

## **Column Descriptions**
- **`Text`**: The user comment or discussion content.  
- **`Polarity`**: Sentiment label indicating the emotional tendency of the text:  
  - `2`: Negative sentiment  
  - `0`: Neutral sentiment  
  - `1`: Positive sentiment  
- **`Platform`**: The source platform of the data, indicating where the comment or discussion originated:  
  - `0`: **GitHub** (Discussions related to open-source projects, Issues, Pull Requests)  
  - `1`: **Jira** (Bug reports, task comments in software development management tools)  
  - `2`: **Mailbox** (Developer communication through emails)  

## **Dataset Distribution**
The dataset consists of data from **GitHub, Jira, and Mailbox**, with different sentiment (`Polarity`) distributions across platforms. It can be used to train and evaluate sentiment classification models to analyze developer emotions on different platforms.  


In [ ]:
import pandas as pd
from tabulate import tabulate

# Load dataset
input_path = '/content/drive/MyDrive/Software_Development_Sentiment_Classification/crossplatform_sf_dataset.csv'
df = pd.read_csv(input_path)

# Compute dataset statistics
total_samples = len(df)
polarity_counts = df["Polarity"].value_counts().sort_index()
platform_counts = df["Platform"].value_counts().sort_index()

# Compute Polarity distribution within each Platform
platform_polarity_counts = df.groupby(["Platform", "Polarity"]).size().unstack().fillna(0)

# Print results with formatting
print("=" * 50)
print(f"📊 Dataset Information: cf-dataset.csv")
print("=" * 50)
print(f"Total Samples: {total_samples}\n")

# Polarity distribution
print("📌 Polarity Distribution:")
print(tabulate(polarity_counts.reset_index(), headers=["Polarity", "Count"], tablefmt="pretty"))
print("\n")

# Platform distribution
print("📌 Platform Distribution:")
print(tabulate(platform_counts.reset_index(), headers=["Platform", "Count"], tablefmt="pretty"))
print("\n")

# Platform-wise Polarity distribution
print("📌 Platform-wise Polarity Distribution:")
print(tabulate(platform_polarity_counts, headers="keys", tablefmt="pretty"))
print("=" * 50)


📊 Dataset Information: cf-dataset.csv
Total Samples: 3227

📌 Polarity Distribution:
+---+----------+-------+
|   | Polarity | Count |
+---+----------+-------+
| 0 |    0     | 1125  |
| 1 |    1     | 1042  |
| 2 |    2     | 1060  |
+---+----------+-------+


📌 Platform Distribution:
+---+----------+-------+
|   | Platform | Count |
+---+----------+-------+
| 0 |    0     | 1079  |
| 1 |    1     | 1054  |
| 2 |    2     | 1094  |
+---+----------+-------+


📌 Platform-wise Polarity Distribution:
+----------+-----+-----+-----+
| Platform |  0  |  1  |  2  |
+----------+-----+-----+-----+
|    0     | 392 | 361 | 326 |
|    1     | 367 | 316 | 371 |
|    2     | 366 | 365 | 363 |
+----------+-----+-----+-----+


# Sentiment Classification

In [ ]:
def seed_torch(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic=True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpu = torch.cuda.device_count()
torch.cuda.get_device_name(0)

'NVIDIA L4'

In [ ]:
# Train
MAX_LEN = 256
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
EPOCHS = 4
WEIGHT_DECAY = 0.01
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


MODELS = [
    (BertForSequenceClassification, BertTokenizer, 'bert-base-cased'),
    (XLNetForSequenceClassification, XLNetTokenizer, 'xlnet-base-cased'),
    (RobertaForSequenceClassification, RobertaTokenizer, 'roberta-base'),
    (AlbertForSequenceClassification, AlbertTokenizer, 'albert-base-v1')
]
MODEL_NAMES = ['bert', 'xlnet', 'roberta', 'albert']

def train_model(train_df, model_save_path, model_select=0):
    seed_torch(42)

    cur_model = MODELS[model_select]
    m_name = MODEL_NAMES[model_select]


    train_df['Polarity'] = train_df['Polarity'].replace({'positive': 1, 'negative': 2, 'neutral': 0})
    tokenizer = cur_model[1].from_pretrained(cur_model[2], do_lower_case=True)

    sentences = train_df.Text.values
    labels = train_df.Polarity.values

    input_ids = []
    attention_masks = []

    for sent in sentences:
        encoded_dict = tokenizer.encode_plus(
            str(sent),
            add_special_tokens=True,
            max_length=MAX_LEN,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )
        input_ids.append(encoded_dict['input_ids'])
        attention_masks.append(encoded_dict['attention_mask'])

    input_ids = torch.cat(input_ids, dim=0)
    attention_masks = torch.cat(attention_masks, dim=0)
    labels = torch.tensor(labels)

    print(f'Training data shape: {input_ids.shape}, {attention_masks.shape}, {labels.shape}')


    train_inputs, val_inputs, train_labels, val_labels = train_test_split(
        input_ids, labels, test_size=0.1, random_state=42)
    train_masks, val_masks, _, _ = train_test_split(
        attention_masks, labels, test_size=0.1, random_state=42)


    train_data = TensorDataset(train_inputs, train_masks, train_labels)
    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=BATCH_SIZE)

    val_data = TensorDataset(val_inputs, val_masks, val_labels)
    val_sampler = SequentialSampler(val_data)
    val_dataloader = DataLoader(val_data, sampler=val_sampler, batch_size=BATCH_SIZE)


    model = cur_model[0].from_pretrained(cur_model[2], num_labels=3)
    model.to(device)


    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


    num_training_steps = EPOCHS * len(train_dataloader)
    lr_scheduler = get_scheduler(
        name="linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
    )


    print("Starting training...")
    best_f1 = 0
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        predictions, true_labels = [], []

        for batch in train_dataloader:
            b_input_ids, b_input_mask, b_labels = [t.to(device) for t in batch]
            optimizer.zero_grad()
            outputs = model(b_input_ids, attention_mask=b_input_mask, labels=b_labels)
            loss, logits = outputs[:2]
            loss.backward()
            optimizer.step()
            lr_scheduler.step()

            total_loss += loss.item()
            predictions.extend(torch.argmax(logits, axis=1).cpu().numpy())
            true_labels.extend(b_labels.cpu().numpy())

        train_acc = accuracy_score(true_labels, predictions)
        print(f"Epoch {epoch+1}: Train Loss: {total_loss / len(train_dataloader):.4f}, Accuracy: {train_acc:.4f}")


        model.eval()
        val_predictions, val_labels = [], []
        with torch.no_grad():
            for batch in val_dataloader:
                b_input_ids, b_input_mask, b_labels = [t.to(device) for t in batch]
                outputs = model(b_input_ids, attention_mask=b_input_mask)
                logits = outputs[0]
                val_predictions.extend(torch.argmax(logits, axis=1).cpu().numpy())
                val_labels.extend(b_labels.cpu().numpy())

        val_acc = accuracy_score(val_labels, val_predictions)
        val_f1 = f1_score(val_labels, val_predictions, average='weighted')
        print(f"Validation Accuracy: {val_acc:.4f}, F1 Score: {val_f1:.4f}")


        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), model_save_path)
            print(f"Best model saved at {model_save_path}")


    print("Final Model Performance on Validation Set:")
    print(classification_report(val_labels, val_predictions, digits=4))
    return model_save_path


In [ ]:

def test_model(test_df, model_saved_path, model_select=0):

  MODELS = [(BertForSequenceClassification,BertTokenizer,'bert-base-cased'),
          (XLNetForSequenceClassification, XLNetTokenizer,'xlnet-base-cased'),
          (RobertaForSequenceClassification, RobertaTokenizer,'roberta-base'),
          (AlbertForSequenceClassification, AlbertTokenizer,'albert-base-v1')
        ]
  MODEL_NAMES = ['bert', 'xlnet', 'Roberta', 'albert']
  seed_torch(42)

  cur_model=MODELS[model_select]
  m_name=MODEL_NAMES[model_select]

  tokenizer = cur_model[1].from_pretrained(cur_model[2], do_lower_case=True)

  begin=time.time()

  test_df['Polarity']=test_df['Polarity'].replace({
      'positive':1,
      'negative':2,
      'neutral':0})


  sentences = test_df.Text.values
  labels = test_df.Polarity.values

  input_ids = []
  attention_masks = []

  for sent in sentences:
      encoded_dict = tokenizer.encode_plus(
                          str(sent),
                          add_special_tokens = True,
                          max_length = MAX_LEN,
                          pad_to_max_length = True,
                          return_attention_mask = True,
                          return_tensors = 'pt',
                    )

      input_ids.append(encoded_dict['input_ids'])
      attention_masks.append(encoded_dict['attention_mask'])

  prediction_inputs = torch.cat(input_ids,dim=0)
  prediction_masks = torch.cat(attention_masks,dim=0)
  prediction_labels = torch.tensor(labels)

  prediction_data = TensorDataset(prediction_inputs, prediction_masks, prediction_labels)
  prediction_sampler = SequentialSampler(prediction_data)
  prediction_dataloader = DataLoader(prediction_data, sampler=prediction_sampler, batch_size=BATCH_SIZE)

  model = cur_model[0].from_pretrained(cur_model[2], num_labels=3)
  model.load_state_dict(torch.load(model_saved_path))
  model.cuda()
  model.eval()

  predictions,true_labels=[],[]

  for batch in prediction_dataloader:
      batch = tuple(t.to(device) for t in batch)
      b_input_ids, b_input_mask, b_labels = batch

      with torch.no_grad():
          outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)
          logits = outputs[0]

      logits = logits.detach().cpu().numpy()
      label_ids = b_labels.to('cpu').numpy()

      predictions.append(logits)
      true_labels.append(label_ids)

  end=time.time()
  print('Prediction used {:.2f} seconds'.format(end - begin))

  flat_predictions = [item for sublist in predictions for item in sublist]
  flat_predictions = np.argmax(flat_predictions, axis=1).flatten()
  flat_true_labels = [item for sublist in true_labels for item in sublist]

  print("Accuracy of {} is: {}".format(m_name, accuracy_score(flat_true_labels,flat_predictions)))

  print(classification_report(flat_true_labels,flat_predictions))


  df_prediction = pd.DataFrame(flat_predictions, columns=['prediction_Polarity'])

  df_combined = pd.concat([test_df, df_prediction], axis=1)

  counts = df_combined['prediction_Polarity'].value_counts()
  print(counts)

  return df_combined

## Train

### Dataset Preparation and Splitting

In this section, we prepare the datasets for training and testing.

- **`crossplatform_sf_dataset_tokenized.csv`**: This is the main dataset used in this study.
- **`so-dataset_tokenized.csv`**: This dataset originates from the research paper *Sentiment Polarity Detection for Software Development*.
- **`gh-dataset_tokenized.csv`**: This dataset is derived from the research paper *GitHub Golden Rule* (*Can We Use SE-specific Sentiment Analysis Tools in a Cross-Platform Setting?*).

### Steps:

1. **Load Datasets**  
   We read the three datasets into Pandas DataFrames.

2. **Split into Training and Testing Sets**  
   - The **GitHub dataset (`df_gh`)** and **Stack Overflow dataset (`df_so`)** are each split into 70% training and 30% testing subsets.  
   - Similarly, the **cross-platform dataset (`df_crossplatform`)** is divided into a 70% training set and a 30% testing set.  
   - The splitting is performed using `train_test_split` with a `random_state` of 42 for reproducibility.

3. **Save Processed Data**  
   - The training and testing subsets are saved as CSV files for further use.



In [ ]:

# Read datasets
input_path = '/content/drive/MyDrive/Software_Development_Sentiment_Classification/crossplatform_sf_dataset_tokenized.csv'
so_input_path = '/content/drive/MyDrive/Software_Development_Sentiment_Classification/so-dataset_tokenized.csv'
gh_input_path = '/content/drive/MyDrive/Software_Development_Sentiment_Classification/gh-dataset_tokenized.csv'

# Load datasets into Pandas DataFrames
df_crossplatform = pd.read_csv(input_path)
df_so = pd.read_csv(so_input_path)
df_gh = pd.read_csv(gh_input_path)

# Split `df_crossplatform` into training (70%) and testing (30%) sets
train_df, test_df = train_test_split(df_crossplatform, test_size=0.3, random_state=42)

# Split GitHub and Stack Overflow datasets into training and testing sets (70% train, 30% test)
train_gh, test_gh = train_test_split(df_gh, test_size=0.3, random_state=42)
train_so, test_so = train_test_split(df_so, test_size=0.3, random_state=42)

# Save all datasets to CSV files for further use

train_df.to_csv('/content/drive/MyDrive/Software_Development_Sentiment_Classification/train_df.csv', index=False)
test_df.to_csv('/content/drive/MyDrive/Software_Development_Sentiment_Classification/test_df.csv', index=False)

train_gh.to_csv('/content/drive/MyDrive/Software_Development_Sentiment_Classification/train_gh.csv', index=False)
train_so.to_csv('/content/drive/MyDrive/Software_Development_Sentiment_Classification/train_so.csv', index=False)
test_gh.to_csv('/content/drive/MyDrive/Software_Development_Sentiment_Classification/test_gh.csv', index=False)
test_so.to_csv('/content/drive/MyDrive/Software_Development_Sentiment_Classification/test_so.csv', index=False)

### Train for Cross-Platform Dataset
We combine three training datasets (`train_df(ours)`, `train_gh`, and `train_so`) into a final dataset for training four different models.

## Training Models  
The following models are trained:  
- **BERT**  
- **XLNet**  
- **RoBERTa**  
- **ALBERT**  

## Training Parameters  
- **MAX_LEN**: `256`  
- **BATCH_SIZE**: `16`  
- **LEARNING_RATE**: `2e-5`  
- **EPOCHS**: `4`  

Each model is trained using the merged dataset and saved for further evaluation.


In [ ]:
# Combine `train_df`, `train_gh`, and `train_so` into the final training dataset
train_df_final = pd.concat([train_df, train_gh, train_so], axis=0, ignore_index=True)
train_df_final.to_csv('/content/drive/MyDrive/Software_Development_Sentiment_Classification/train_df_final.csv', index=False)

# Define the list of model names to be trained
MODEL_NAMES = ['bert', 'xlnet', 'Roberta', 'albert']

# Train each model and save the trained model files
for i, model_name in enumerate(MODEL_NAMES):
    model_save_path = f"/content/drive/MyDrive/Software_Development_Sentiment_Classification/{model_name}_model"
    print(f"Training {model_name} model...")
    train_model(train_df_final, model_save_path, model_select=i)


Training bert model...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training data shape: torch.Size([4068, 256]), torch.Size([4068, 256]), torch.Size([4068])
Starting training...


/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1: Train Loss: 0.5891, Accuracy: 0.7577
Validation Accuracy: 0.8673, F1 Score: 0.8675
Best model saved at /content/drive/MyDrive/Software_Development_Sentiment_Classification/bert_model
Epoch 2: Train Loss: 0.2148, Accuracy: 0.9312
Validation Accuracy: 0.8771, F1 Score: 0.8772
Best model saved at /content/drive/MyDrive/Software_Development_Sentiment_Classification/bert_model
Epoch 3: Train Loss: 0.1029, Accuracy: 0.9683
Validation Accuracy: 0.8845, F1 Score: 0.8841
Best model saved at /content/drive/MyDrive/Software_Development_Sentiment_Classification/bert_model
Epoch 4: Train Loss: 0.0526, Accuracy: 0.9869
Validation Accuracy: 0.9017, F1 Score: 0.9015
Best model saved at /content/drive/MyDrive/Software_Development_Sentiment_Classification/bert_model
Final Model Performance on Validation Set:
              precision    recall  f1-score   support

           0     0.9294    0.8587    0.8927       184
           1     0.8739    0.9369    0.9043       111
           2     0.8898   

...

Training data shape: torch.Size([4068, 256]), torch.Size([4068, 256]), torch.Size([4068])
Starting training...


/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1: Train Loss: 0.6244, Accuracy: 0.7228
Validation Accuracy: 0.8698, F1 Score: 0.8697
Best model saved at /content/drive/MyDrive/Software_Development_Sentiment_Classification/Roberta_model
Epoch 2: Train Loss: 0.2585, Accuracy: 0.9123
Validation Accuracy: 0.8747, F1 Score: 0.8745
Best model saved at /content/drive/MyDrive/Software_Development_Sentiment_Classification/Roberta_model
Epoch 3: Train Loss: 0.1612, Accuracy: 0.9506
Validation Accuracy: 0.8968, F1 Score: 0.8968
Best model saved at /content/drive/MyDrive/Software_Development_Sentiment_Classification/Roberta_model
Epoch 4: Train Loss: 0.0974, Accuracy: 0.9727
Validation Accuracy: 0.8968, F1 Score: 0.8969
Best model saved at /content/drive/MyDrive/Software_Development_Sentiment_Classification/Roberta_model
Final Model Performance on Validation Set:
              precision    recall  f1-score   support

           0     0.9302    0.8696    0.8989       184
           1     0.8803    0.9279    0.9035       111
           2  

Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at albert-base-v1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Starting training...
Epoch 1: Train Loss: 0.6810, Accuracy: 0.7110
Validation Accuracy: 0.8305, F1 Score: 0.8332
Best model saved at /content/drive/MyDrive/Software_Development_Sentiment_Classification/albert_model
Epoch 2: Train Loss: 0.3244, Accuracy: 0.8913
Validation Accuracy: 0.8550, F1 Score: 0.8550
Best model saved at /content/drive/MyDrive/Software_Development_Sentiment_Classification/albert_model
Epoch 3: Train Loss: 0.2071, Accuracy: 0.9306
Validation Accuracy: 0.8550, F1 Score: 0.8548
Epoch 4: Train Loss: 0.1405, Accuracy: 0.9566
Validation Accuracy: 0.8526, F1 Score: 0.8524
Final Model Performance on Validation Set:
              precision    recall  f1-score   support

           0     0.8715    0.8478    0.8595       184
           1     0.8547    0.9009    0.8772       111
           2     0.8198    0.8125    0.8161       112

    accuracy                         0.8526       407
   macro avg     0.8487    0.8537    0.8509       407
weighted avg     0.8527    0.8526    0

### Train for existing dataset on Bert
In this section, we train the **BERT** model using the existing **GitHub** and **Stack Overflow** datasets.

## Training Data  
The training dataset consists of:  
- **GitHub Data** (`train_gh`)  
- **Stack Overflow Data** (`train_so`)  

## Training Parameters  
- **MAX_LEN**: `256`  
- **BATCH_SIZE**: `16`  
- **LEARNING_RATE**: `2e-5`  
- **EPOCHS**: `4`  

The trained **BERT** model will be saved for further evaluation.

In [ ]:
# Train the model for github-golden-rule and stackoverflow on Bert

MODEL_NAMES = ['bert']
for dataset_name, train_df in [('SO', train_so), ('GH', train_gh)]:
    for i, model_name in enumerate(MODEL_NAMES):
        model_save_path = f"/content/drive/MyDrive/Software_Development_Sentiment_Classification/{dataset_name}_{model_name}_model"
        print(f"Training {model_name} model on {dataset_name} dataset...")
        train_model(train_df, model_save_path, model_select=i)


Training bert model on SO dataset...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Training data shape: torch.Size([315, 256]), torch.Size([315, 256]), torch.Size([315])
Starting training...
Epoch 1: Train Loss: 0.7896, Accuracy: 0.7032
Validation Accuracy: 0.8438, F1 Score: 0.7722
Best model saved at /content/drive/MyDrive/Software_Development_Sentiment_Classification/SO_bert_model
Epoch 2: Train Loss: 0.5530, Accuracy: 0.8163
Validation Accuracy: 0.8438, F1 Score: 0.7722
Epoch 3: Train Loss: 0.4831, Accuracy: 0.8163
Validation Accuracy: 0.8438, F1 Score: 0.7722
Epoch 4: Train Loss: 0.4383, Accuracy: 0.8163
Validation Accuracy: 0.8438, F1 Score: 0.7722
Final Model Performance on Validation Set:
              precision    recall  f1-score   support

           0     0.8438    1.0000    0.9153        27
           1     0.0000    0.0000    0.0000         4
           2     0.0000    0.0000    0.0000         1

    accuracy                         0.8438        32
   macro avg     0.2812    0.3333    0.3051        32
weighted avg     0.7119    0.8438    0.7722        3

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
Some weights of BertForSequenceClassification wer

Training data shape: torch.Size([1495, 256]), torch.Size([1495, 256]), torch.Size([1495])
Starting training...


/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1: Train Loss: 0.8448, Accuracy: 0.6141
Validation Accuracy: 0.7933, F1 Score: 0.7983
Best model saved at /content/drive/MyDrive/Software_Development_Sentiment_Classification/GH_bert_model
Epoch 2: Train Loss: 0.3207, Accuracy: 0.8900
Validation Accuracy: 0.8733, F1 Score: 0.8741
Best model saved at /content/drive/MyDrive/Software_Development_Sentiment_Classification/GH_bert_model
Epoch 3: Train Loss: 0.1529, Accuracy: 0.9532
Validation Accuracy: 0.8867, F1 Score: 0.8851
Best model saved at /content/drive/MyDrive/Software_Development_Sentiment_Classification/GH_bert_model
Epoch 4: Train Loss: 0.0991, Accuracy: 0.9762
Validation Accuracy: 0.8800, F1 Score: 0.8783
Final Model Performance on Validation Set:
              precision    recall  f1-score   support

           0     0.8868    0.8246    0.8545        57
           1     0.8846    1.0000    0.9388        46
           2     0.8667    0.8298    0.8478        47

    accuracy                         0.8800       150
   macro

## Test

### Test df_crossplatform on 4 models and 3 platforms (Table 3.2)
In this section, we evaluate the four trained **cross-platform sentiment classification models** on our **cross-platform sentiment dataset**.

## Evaluation Metrics  
We will assess:  
1. **Overall model performance** across all platforms.  
2. **Platform-specific performance** for each model on:  
   - **GitHub**  
   - **Jira**  
   - **Mailbox**  

## Results  
The evaluation will print:  
- **Overall accuracy** of each model.  
- **Performance breakdown per platform** for each model.  








In [ ]:

# Load test dataset
test_df = pd.read_csv('/content/drive/MyDrive/Software_Development_Sentiment_Classification/test_df.csv')

MODEL_NAMES = ['bert', 'xlnet', 'Roberta', 'albert']
model_results = {}

# Define platform mapping
platforms = {0: "GitHub", 1: "Jira", 2: "Mailbox"}

# Evaluate each model
for i, model_name in enumerate(MODEL_NAMES):
    model_path = f"/content/drive/MyDrive/Software_Development_Sentiment_Classification/{model_name}_model"
    print(f"Evaluating {model_name} model...for overall platform")

    # Get overall accuracy
    overall_accuracy = test_model(test_df, model_path, model_select=i)

    # Evaluate accuracy per platform
    platform_accuracies = {}
    for platform_id, platform_name in platforms.items():
        test_df_platform = test_df[test_df["Platform"] == platform_id]
        if not test_df_platform.empty:
            print(f"Evaluating {model_name} model...for {platform_name} platform")
            accuracy = test_model(test_df_platform, model_path, model_select=i)
            platform_accuracies[platform_name] = accuracy
        else:
            platform_accuracies[platform_name] = "No data"





Evaluating bert model...for overall platform


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:2673: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:2673: FutureWarning

Prediction used 9.38 seconds
Accuracy of bert is: 0.9422084623323014
              precision    recall  f1-score   support

           0       0.95      0.91      0.93       329
           1       0.94      0.97      0.95       318
           2       0.94      0.95      0.94       322

    accuracy                           0.94       969
   macro avg       0.94      0.94      0.94       969
weighted avg       0.94      0.94      0.94       969

prediction_Polarity
2    327
1    326
0    316
Name: count, dtype: int64
Evaluating bert model...for GitHub platform


/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:2673: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-4-486b4f5d4361>:53: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. 

Prediction used 3.93 seconds
Accuracy of bert is: 0.956140350877193
              precision    recall  f1-score   support

           0       0.96      0.94      0.95       128
           1       0.97      0.96      0.96       122
           2       0.94      0.98      0.96        92

    accuracy                           0.96       342
   macro avg       0.95      0.96      0.96       342
weighted avg       0.96      0.96      0.96       342

prediction_Polarity
0.0    125
1.0    121
2.0     96
Name: count, dtype: int64
Evaluating bert model...for Jira platform


...

Prediction used 3.99 seconds
Accuracy of albert is: 0.9269005847953217
              precision    recall  f1-score   support

           0       0.93      0.88      0.90       128
           1       0.97      0.96      0.96       122
           2       0.87      0.96      0.91        92

    accuracy                           0.93       342
   macro avg       0.92      0.93      0.93       342
weighted avg       0.93      0.93      0.93       342

prediction_Polarity
1.0    121
0.0    120
2.0    101
Name: count, dtype: int64
Evaluating albert model...for Jira platform


<ipython-input-4-486b4f5d4361>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['Polarity']=test_df['Polarity'].replace({
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:2673: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest

Prediction used 3.30 seconds
Accuracy of albert is: 0.89198606271777
              precision    recall  f1-score   support

           0       0.95      0.75      0.84        93
           1       0.91      0.94      0.93        86
           2       0.85      0.97      0.91       108

    accuracy                           0.89       287
   macro avg       0.90      0.89      0.89       287
weighted avg       0.90      0.89      0.89       287

prediction_Polarity
2.0    124
1.0     89
0.0     74
Name: count, dtype: int64
Evaluating albert model...for Mailbox platform


<ipython-input-4-486b4f5d4361>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['Polarity']=test_df['Polarity'].replace({
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:2673: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest

Prediction used 4.12 seconds
Accuracy of albert is: 0.8352941176470589
              precision    recall  f1-score   support

           0       0.83      0.72      0.77       108
           1       0.85      0.94      0.89       110
           2       0.82      0.84      0.83       122

    accuracy                           0.84       340
   macro avg       0.84      0.83      0.83       340
weighted avg       0.83      0.84      0.83       340

prediction_Polarity
2.0    125
1.0    121
0.0     94
Name: count, dtype: int64


### Generalization Performance of the Model (Table 3.3)
In this section, we evaluate the **Bert-CP** model's **generalization performance** on the existing datasets:  
- **GitHub Golden Rule Dataset**  
- **Stack Overflow Dataset**  

We will also compare the performance of the **BERT model** trained on **GitHub Golden Rule** and **Stack Overflow** datasets, with a focus on **cross-platform performance**. This comparison aims to validate the **superiority** of our model.

## Evaluation Process  
- **Bert-CP Model Evaluation**: We test the **Bert-CP** model on the **GitHub Golden Rule** and **Stack Overflow** datasets.
- **Cross-Platform Comparison**: We compare the performance of models trained on **GitHub Golden Rule** and **Stack Overflow** datasets across multiple platforms using the **BERT model**.

## Goals  
- To assess the **generalization** of the **Bert-CP** model across different datasets.
- To highlight the **superiority** of our cross-platform model over dataset-specific models.

In [ ]:


# Load test datasets
test_gh = pd.read_csv('/content/drive/MyDrive/Software_Development_Sentiment_Classification/test_gh.csv')
test_so = pd.read_csv('/content/drive/MyDrive/Software_Development_Sentiment_Classification/test_so.csv')

# Define model paths
bert_model_path = "/content/drive/MyDrive/Software_Development_Sentiment_Classification/bert_model"
gh_bert_model_path = "/content/drive/MyDrive/Software_Development_Sentiment_Classification/GH_bert_model"
so_bert_model_path = "/content/drive/MyDrive/Software_Development_Sentiment_Classification/SO_bert_model"

# Store results
model_results = {}

# 1. Validate bert_model on test_gh and test_so
print("Evaluating bert_model on GitHub test dataset...")
bert_on_gh = test_model(test_gh, bert_model_path, model_select=0)

print("Evaluating bert_model on Stack Overflow test dataset...")
bert_on_so = test_model(test_so, bert_model_path, model_select=0)

model_results["bert_model"] = {
    "test_gh Accuracy": bert_on_gh,
    "test_so Accuracy": bert_on_so
}

# 2. Validate GH_bert_model on test_so
print("Evaluating GH_bert_model on Stack Overflow test dataset...")
gh_bert_on_so = test_model(test_so, gh_bert_model_path, model_select=0)

model_results["GH_bert_model"] = {
    "test_so Accuracy": gh_bert_on_so
}

# 3. Validate SO_bert_model on test_gh
print("Evaluating SO_bert_model on GitHub test dataset...")
so_bert_on_gh = test_model(test_gh, so_bert_model_path, model_select=0)

model_results["SO_bert_model"] = {
    "test_gh Accuracy": so_bert_on_gh
}


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


Evaluating bert_model on GitHub test dataset...


/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:2673: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-4-486b4f5d4361>:53: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. 

Prediction used 7.34 seconds
Accuracy of bert is: 0.8829953198127926
              precision    recall  f1-score   support

           0       0.91      0.86      0.88       267
           1       0.89      0.92      0.91       170
           2       0.85      0.88      0.86       204

    accuracy                           0.88       641
   macro avg       0.88      0.89      0.88       641
weighted avg       0.88      0.88      0.88       641

prediction_Polarity
0    252
2    213
1    176
Name: count, dtype: int64
Evaluating bert_model on Stack Overflow test dataset...


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:2673: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ber

...

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:2673: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-4-486b4f5d4361>:53: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. 

Prediction used 2.21 seconds
Accuracy of bert is: 0.8148148148148148
              precision    recall  f1-score   support

           0       0.85      0.95      0.89       110
           1       0.50      0.27      0.35        11
           2       0.50      0.21      0.30        14

    accuracy                           0.81       135
   macro avg       0.62      0.48      0.52       135
weighted avg       0.78      0.81      0.79       135

prediction_Polarity
0    123
2      6
1      6
Name: count, dtype: int64
Evaluating SO_bert_model on GitHub test dataset...


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:2673: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ber

Prediction used 6.62 seconds
Accuracy of bert is: 0.4165366614664587
              precision    recall  f1-score   support

           0       0.42      1.00      0.59       267
           1       0.00      0.00      0.00       170
           2       0.00      0.00      0.00       204

    accuracy                           0.42       641
   macro avg       0.14      0.33      0.20       641
weighted avg       0.17      0.42      0.24       641

prediction_Polarity
0    641
Name: count, dtype: int64


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
